# Mouse PBMC dataset: sanity-check the numbers quoted in the paper

Recomputes, directly from the on-disk data, every number used in the paper's
Section 6 and Appendix B.1: raw cell/gene counts, batch sizes, the
cell-type x batch contingency table, the gene-filtering count, and the
final HVG-selected matrix shape. The contingency table is cross-checked
cell-by-cell against **Table 1** of Cao & Ma, "MoDaH achieves rate optimal
batch correction" (arXiv:2512.09259), the paper the raw PBMC subset and
the six-batch mouse PBMC dataset are drawn from -- their Table 1 reports
the exact same contingency table for this dataset (Han et al. 2018 Mouse
Cell Atlas, GEO GSE108097, restricted to the `Peripheral_Blood` tissue
subset), so an exact match here is a strong independent confirmation that
we loaded and are reporting the same data.

Two files are inspected:
- The raw PBMC subset h5ad (raw counts, pre-`preprocess.py`) -- see
  `preprocess.py` for where to obtain this.
- `data/mouse_pbmc_hvg_lognorm.h5ad` -- the log-normalized, top-2000-HVG
  matrix that `preprocess.py` produces from the file above, and that
  `common.py`/`run_mouse_pbmc_comparison.py` load for the Table 1 comparison.

In [ ]:
from __future__ import annotations

import os

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

_THIS_DIR = os.path.dirname(os.path.abspath("__file__"))
# EDIT: point this at your local copy of the raw PBMC subset h5ad (see preprocess.py).
RAW_PATH = os.path.join(_THIS_DIR, "..", "..", "raw_data", "pbmc_subset.h5ad")
PROCESSED_PATH = os.path.join(_THIS_DIR, "data", "mouse_pbmc_hvg_lognorm.h5ad")

raw = sc.read_h5ad(RAW_PATH)
processed = sc.read_h5ad(PROCESSED_PATH)
print("raw:", raw.shape)
print("processed:", processed.shape)

## 1. Raw data: cell/gene counts, tissue restriction, batch and cell-type columns

In [ ]:
print(f"Raw count matrix: {raw.n_obs:,} cells x {raw.n_vars:,} genes")
print("Tissue values present:", raw.obs["Tissue"].unique().tolist())

X_head = raw.X[:500]
X_head = X_head.toarray() if sp.issparse(X_head) else np.asarray(X_head)
print("First 500 rows all non-negative integers (i.e. raw counts)?",
      bool(np.all(X_head >= 0) and np.allclose(X_head, np.round(X_head))))

print("\nobs columns:", raw.obs.columns.tolist())
print("\n'dataset' (batch) value counts:")
print(raw.obs["dataset"].value_counts())
print("\n'cell_type_combined' (coarsened cell type) value counts:")
print(raw.obs["cell_type_combined"].value_counts())
print(f"\nNumber of distinct cell types (K): {raw.obs['cell_type_combined'].nunique()}")

print("\nHow the 9 broad cell types are derived from the atlas's finer 'Annotation' labels")
print("(cell_type_broad = Annotation.split('_')[0]) -- a few examples:")
print(raw.obs[["Annotation", "cell_type_broad"]].drop_duplicates().sort_values("cell_type_broad").head(15))

## 2. Cell type x batch contingency table -- and cross-check against MoDaH Table 1

In [ ]:
BATCH_ORDER = [f"PeripheralBlood_{i}" for i in range(1, 7)]
CELL_TYPE_ORDER = [
    "B cell", "Basophil", "Dendritic cell", "Erythroblast",
    "Macrophage", "Monocyte", "NK cell", "Neutrophil", "T cell",
]

contingency = (
    raw.obs.groupby(["dataset", "cell_type_combined"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(index=BATCH_ORDER, columns=CELL_TYPE_ORDER)
)
display_table = contingency.T.copy()
display_table["Total"] = display_table.sum(axis=1)
display_table.loc["Total"] = display_table.sum(axis=0)
print("Recomputed cell type (rows) x batch (columns) contingency table:")
print(display_table)

In [ ]:
# Table 1 of Cao & Ma, arXiv:2512.09259 ("Cell counts by cell type annotations
# and batches in the mouse PBMC dataset"), transcribed verbatim for comparison.
MODAH_TABLE_1 = pd.DataFrame(
    {
        "PeripheralBlood_1": [82, 0, 0, 0, 39, 0, 13, 14, 135],
        "PeripheralBlood_2": [225, 55, 71, 93, 136, 358, 12, 1469, 47],
        "PeripheralBlood_3": [1005, 3, 1, 17, 686, 0, 163, 346, 980],
        "PeripheralBlood_4": [14, 0, 0, 0, 82, 0, 20, 0, 19],
        "PeripheralBlood_5": [21, 0, 2, 0, 187, 1, 74, 17, 50],
        "PeripheralBlood_6": [64, 0, 0, 4, 50, 0, 37, 46, 457],
    },
    index=CELL_TYPE_ORDER,
)

diff = contingency.T - MODAH_TABLE_1
n_mismatches = int((diff != 0).to_numpy().sum())
print(f"Cells that differ from MoDaH paper's Table 1: {n_mismatches} / {diff.size}")
if n_mismatches:
    print(diff.loc[(diff != 0).any(axis=1)])
else:
    print("Exact match on all 54 cells (9 cell types x 6 batches).")

## 3. Batch-size spread, and which cell types are missing from which batch

In [ ]:
batch_sizes = contingency.sum(axis=1).reindex(BATCH_ORDER)
print("Batch sizes:")
print(batch_sizes)
print(f"\nTotal cells: {int(batch_sizes.sum()):,}")
print(f"Smallest / largest batch: {int(batch_sizes.min())} / {int(batch_sizes.max())} "
      f"({batch_sizes.max() / batch_sizes.min():.1f}x spread)")

print("\nMissing cell types by batch (0 cells of that type in that batch):")
for batch in BATCH_ORDER:
    missing = contingency.loc[batch][contingency.loc[batch] == 0].index.tolist()
    tag = f"{len(missing)} missing" if missing else "none missing"
    print(f"  {batch} (n={int(batch_sizes[batch])}): {tag}" + (f" -- {missing}" if missing else ""))

n_batches_missing_something = sum(
    1 for batch in BATCH_ORDER if (contingency.loc[batch] == 0).any()
)
print(f"\n{n_batches_missing_something} / {len(BATCH_ORDER)} batches are missing at least one cell type.")

## 4. Reproduce the preprocessing gene-filtering step independently

Recomputes `sc.pp.filter_genes(min_cells=10)` directly from the raw counts,
without relying on `preprocess.py`'s own printed log, then checks the result
matches both that log and the final processed file's provenance.

In [ ]:
raw_for_filter = raw.copy()
n_genes_before = raw_for_filter.n_vars
sc.pp.filter_genes(raw_for_filter, min_cells=10)
n_genes_after_filter = raw_for_filter.n_vars
print(f"Genes before filtering: {n_genes_before:,}")
print(f"Genes with >= 10 expressing cells (kept): {n_genes_after_filter:,}")
print(f"Genes removed (< 10 expressing cells): {n_genes_before - n_genes_after_filter:,}")

print(f"\nFinal processed matrix (after batch-aware top-2000 HVG selection): {processed.shape}")
print("processed.var columns (confirms HVG selection metadata is present):", processed.var.columns.tolist())

## 5. Summary -- numbers as quoted in the paper draft

In [ ]:
print("=" * 72)
print("SUMMARY (compare against the paper's Section 6 / Appendix B.1)")
print("=" * 72)
print(f"Raw:        {raw.n_obs:,} cells x {raw.n_vars:,} genes, {raw.obs['cell_type_combined'].nunique()} cell types")
print(f"Batches:    {len(BATCH_ORDER)} -- sizes {batch_sizes.tolist()}")
print(f"Gene filter (min_cells=10): {n_genes_before:,} -> {n_genes_after_filter:,} genes")
print(f"Final:      {processed.n_obs:,} cells x {processed.n_vars:,} genes (top HVGs)")
print(f"Batches missing >=1 cell type: {n_batches_missing_something} / {len(BATCH_ORDER)}")
print(f"Batch-size spread: {batch_sizes.max() / batch_sizes.min():.1f}x (min {int(batch_sizes.min())}, max {int(batch_sizes.max())})")